# Coleta de dados - SAEB 2023

Microdados do SAEB 2023, do INEP (dados abertos).

In [1]:
import os
import urllib.request
import zipfile
import pandas as pd

## Baixando os dados
Na primeira vez baixa o zip do INEP (uns 700 MB) e coloca as tabelas .csv na pasta `tabelas_csv`.

In [2]:
url = "https://download.inep.gov.br/microdados/microdados_saeb_2023.zip"

if not os.path.exists("tabelas_csv/TS_ESCOLA.csv"):
    if not os.path.exists("microdados_saeb_2023.zip"):
        print("baixando o zip do INEP... (demora um pouco)")
        # baixa com outro nome e so renomeia no final, assim um download pela metade nao conta
        urllib.request.urlretrieve(url, "baixando.zip")
        os.rename("baixando.zip", "microdados_saeb_2023.zip")

    # tira so os .csv de dentro do zip e joga na pasta tabelas_csv
    print("extraindo as tabelas .csv...")
    os.makedirs("tabelas_csv", exist_ok=True)
    z = zipfile.ZipFile("microdados_saeb_2023.zip")
    for nome in z.namelist():
        if nome.endswith(".csv"):
            with open("tabelas_csv/" + os.path.basename(nome), "wb") as f:
                f.write(z.read(nome))
    z.close()

print("tabelas .csv prontas!\n")

baixando o zip do INEP... (demora um pouco)
extraindo as tabelas .csv...
tabelas .csv prontas!



## Tabela de escolas

In [3]:
escolas = pd.read_csv("tabelas_csv/TS_ESCOLA.csv", sep=";", encoding="latin1")
print("tabela escolas:", escolas.shape)

tabela escolas: (70151, 137)


## Alunos do 9º ano
O arquivo é muito grande, então pegamos só algumas colunas.

In [4]:
colunas = ["ID_ALUNO", "ID_ESCOLA", "ID_UF", "IN_PUBLICA", "ID_LOCALIZACAO",
           "IN_PREENCHIMENTO_QUESTIONARIO", "INSE_ALUNO", "PROFICIENCIA_MT_SAEB",
           "TX_RESP_Q01", "TX_RESP_Q02", "TX_RESP_Q08", "TX_RESP_Q19",
           "TX_RESP_Q20", "TX_RESP_Q21d"]
alunos = pd.read_csv("tabelas_csv/TS_ALUNO_9EF.csv", sep=";", encoding="latin1", usecols=colunas)
print("tabela alunos 9 ano:", alunos.shape)

tabela alunos 9 ano: (2502907, 14)


## Valores vazios

In [5]:
print("\nporcentagem de valores vazios:")
print(alunos.isna().mean() * 100)


porcentagem de valores vazios:
ID_UF                             0.000000
ID_ESCOLA                         0.000000
IN_PUBLICA                        0.000000
ID_LOCALIZACAO                    0.000000
ID_ALUNO                          0.000000
PROFICIENCIA_MT_SAEB             16.580840
IN_PREENCHIMENTO_QUESTIONARIO     0.000000
INSE_ALUNO                       20.536041
TX_RESP_Q01                       0.000000
TX_RESP_Q02                       0.000000
TX_RESP_Q08                       0.000000
TX_RESP_Q19                       0.000000
TX_RESP_Q20                       0.000000
TX_RESP_Q21d                      0.000000
dtype: float64


## Q20: "alguma vez você abandonou a escola?"
A = nunca, B = uma vez, C = duas ou mais, . = branco, * = nulo

In [6]:
print("\nrespostas da Q20:")
print(alunos["TX_RESP_Q20"].value_counts())


respostas da Q20:
TX_RESP_Q20
A    1871927
.     514025
B      96342
C      19647
*        966
Name: count, dtype: int64


## Juntando alunos com escolas

In [7]:
dados = alunos.merge(escolas[["ID_ESCOLA", "NIVEL_SOCIO_ECONOMICO"]], on="ID_ESCOLA", how="left")
print("\nalunos que acharam a escola:", dados["NIVEL_SOCIO_ECONOMICO"].notna().mean() * 100, "%")

dados.to_csv("alunos_9ano.csv", index=False)
print("salvo em alunos_9ano.csv")


alunos que acharam a escola: 97.77203068272213 %
salvo em alunos_9ano.csv
